In [16]:
"""
S&P 500 Stock Price Prediction Model Training
Run this first to create and save your model
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
import xgboost as xgb

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Stock data
import pickle
import os

In [17]:
print("="*60)
print("TRAINING S&P 500 PRICE PREDICTION MODEL")
print("="*60)

# Create directories
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

TRAINING S&P 500 PRICE PREDICTION MODEL


#### ============================================
## 1. DATA COLLECTION
### ============================================

In [18]:
# ============================================
# 1. DATA COLLECTION
# ============================================
print("\n1. COLLECTING DATA...")
print("-"*40)
from datetime import datetime, timedelta
from pandas_datareader import data as pdr
API_KEY = "MK7XG38N5QTVAFEI"  # 

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'LLY', 'V', 'JPM']
print(f"Training on {len(tickers)} stocks: {', '.join(tickers)}")

# Use 5 years of data
end = datetime.now()
start = end - timedelta(days=5*365)

all_data = {}

for ticker in tickers:
    try:
        df = pdr.DataReader(ticker, 'stooq', start, end)
        df.sort_index(inplace=True)
        
        if len(df) > 0:
            all_data[ticker] = df
            print(f"   {ticker}: {len(df)} days")
        else:
            print(f"   {ticker}: No data")
    except Exception as e:
        print(f"   {ticker}: Failed — {e}")


1. COLLECTING DATA...
----------------------------------------
Training on 9 stocks: AAPL, MSFT, GOOGL, AMZN, NVDA, META, LLY, V, JPM
   AAPL: 1255 days
   MSFT: 1255 days
   GOOGL: 1255 days
   AMZN: 1255 days
   NVDA: 1255 days
   META: 1255 days
   LLY: 1255 days
   V: 1255 days
   JPM: 1255 days



### 2. DATA PREPROCESSING


In [19]:
# ============================================
# 2. DATA PREPROCESSING (Fixed)
# ============================================
print("\n2. PREPROCESSING DATA...")
print("-"*40)

def preprocess_data(df):
    """Clean and add technical indicators WITHOUT future leakage"""
    df = df.copy()
    
    # Remove duplicates and sort
    df = df[~df.index.duplicated(keep='first')]
    df.sort_index(inplace=True)
    
    # Handle missing values using only past data
    df.fillna(method='ffill', inplace=True)
    
    # Create features using ONLY past data (shifted to avoid leakage)
    
    # Lagged prices (use yesterday's data to predict today)
    df['Close_Lag1'] = df['Close'].shift(1)
    df['Close_Lag2'] = df['Close'].shift(2)
    df['Close_Lag3'] = df['Close'].shift(3)
    df['Close_Lag5'] = df['Close'].shift(5)
    
    # Returns (using lagged data)
    df['Returns_Lag1'] = df['Close'].pct_change().shift(1)
    df['Returns_Lag2'] = df['Close'].pct_change().shift(2)
    
    # Moving averages (using only past data)
    df['SMA_10'] = df['Close'].rolling(window=10, min_periods=10).mean().shift(1)
    df['SMA_20'] = df['Close'].rolling(window=20, min_periods=20).mean().shift(1)
    df['SMA_50'] = df['Close'].rolling(window=50, min_periods=50).mean().shift(1)
    
    # Price ranges (using only past data)
    df['High_Low_Range'] = (df['High'] - df['Low']).shift(1)
    df['Open_Close_Range'] = (df['Close'] - df['Open']).shift(1)
    
    # Volume indicators (using only past data)
    df['Volume_SMA'] = df['Volume'].rolling(window=5, min_periods=5).mean().shift(1)
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA']
    
    # Volatility (using only past data)
    df['Volatility_10'] = df['Close'].pct_change().rolling(window=10).std().shift(1)
    df['Volatility_20'] = df['Close'].pct_change().rolling(window=20).std().shift(1)
    
    # Remove rows with NaN values
    df.dropna(inplace=True)
    
    return df

# Process each stock separately
processed_data = {}
for ticker, df in all_data.items():
    processed_data[ticker] = preprocess_data(df)
    print(f"   {ticker}: {len(processed_data[ticker])} samples after preprocessing")


2. PREPROCESSING DATA...
----------------------------------------
   AAPL: 1205 samples after preprocessing
   MSFT: 1205 samples after preprocessing
   GOOGL: 1205 samples after preprocessing
   AMZN: 1205 samples after preprocessing
   NVDA: 1205 samples after preprocessing
   META: 1205 samples after preprocessing
   LLY: 1205 samples after preprocessing
   V: 1205 samples after preprocessing
   JPM: 1205 samples after preprocessing


### 3. FEATURE SELECTION

In [20]:
# ============================================
# 3. FEATURE SELECTION
# ============================================
print("\n3. SELECTING FEATURES...")
print("-"*40)

# Define features (exclude price columns that would cause leakage)
exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Close_Lag1']  # Keep Close_Lag1 as feature
feature_columns = [col for col in processed_data[tickers[0]].columns if col not in exclude_cols]
target_column = 'Close'  # We want to predict the actual close price

print(f"Features ({len(feature_columns)}): {feature_columns[:10]}...")


3. SELECTING FEATURES...
----------------------------------------
Features (14): ['Close_Lag2', 'Close_Lag3', 'Close_Lag5', 'Returns_Lag1', 'Returns_Lag2', 'SMA_10', 'SMA_20', 'SMA_50', 'High_Low_Range', 'Open_Close_Range']...


### 4. TRAIN/TEST SPLIT (Fixed - By Stock)

In [21]:
# ============================================
# 4. TRAIN/TEST SPLIT (Fixed - By Stock)
# ============================================
print("\n4. SPLITTING DATA...")
print("-"*40)

# Process each stock separately and maintain temporal order
X_train_list = []
X_test_list = []
y_train_list = []
y_test_list = []

# Use 80% for training, 20% for testing for each stock
train_ratio = 0.8

for ticker, df in processed_data.items():
    # Extract features and target
    X_stock = df[feature_columns].values
    y_stock = df[target_column].values
    
    # Scale per stock to avoid look-ahead bias
    scaler_X_stock = MinMaxScaler()
    scaler_y_stock = MinMaxScaler()
    
    X_stock_scaled = scaler_X_stock.fit_transform(X_stock)
    y_stock_scaled = scaler_y_stock.fit_transform(y_stock.reshape(-1, 1)).ravel()
    
    # Split temporally (no shuffling)
    split_idx = int(len(X_stock_scaled) * train_ratio)
    
    X_train_list.append(X_stock_scaled[:split_idx])
    X_test_list.append(X_stock_scaled[split_idx:])
    y_train_list.append(y_stock_scaled[:split_idx])
    y_test_list.append(y_stock_scaled[split_idx:])
    
    print(f"   {ticker}: Train={split_idx}, Test={len(X_stock_scaled)-split_idx}")

# Combine all stocks
X_train = np.vstack(X_train_list)
X_test = np.vstack(X_test_list)
y_train = np.concatenate(y_train_list)
y_test = np.concatenate(y_test_list)

print(f"\nTotal training samples: {len(X_train)}")
print(f"Total testing samples: {len(X_test)}")

# Fit global scalers (optional - for saving)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
scaler_X.fit(np.vstack([X_train, X_test]))
scaler_y.fit(np.concatenate([y_train, y_test]).reshape(-1, 1))


4. SPLITTING DATA...
----------------------------------------
   AAPL: Train=964, Test=241
   MSFT: Train=964, Test=241
   GOOGL: Train=964, Test=241
   AMZN: Train=964, Test=241
   NVDA: Train=964, Test=241
   META: Train=964, Test=241
   LLY: Train=964, Test=241
   V: Train=964, Test=241
   JPM: Train=964, Test=241

Total training samples: 8676
Total testing samples: 2169


MinMaxScaler()

### 5. TRAIN MULTIPLE MODELS

In [22]:
# ============================================
# 5. TRAIN MULTIPLE MODELS
# ============================================
print("\n5. TRAINING MODELS...")
print("-"*40)

results = {}

# 1. Linear Regression
print("\nLinear Regression...")
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
results['Linear Regression'] = {
    'model': lr,
    'r2': r2_score(y_test, lr_pred),
    'mse': mean_squared_error(y_test, lr_pred),
    'mae': mean_absolute_error(y_test, lr_pred)
}
print(f"   R² Score: {results['Linear Regression']['r2']:.4f}")
print(f"   MAE: {results['Linear Regression']['mae']:.4f}")



5. TRAINING MODELS...
----------------------------------------

Linear Regression...
   R² Score: 0.9760
   MAE: 0.0178


In [23]:
# 2. Random Forest
print("\nRandom Forest...")
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results['Random Forest'] = {
    'model': rf,
    'r2': r2_score(y_test, rf_pred),
    'mse': mean_squared_error(y_test, rf_pred),
    'mae': mean_absolute_error(y_test, rf_pred)
}
print(f"   R² Score: {results['Random Forest']['r2']:.4f}")
print(f"   MAE: {results['Random Forest']['mae']:.4f}")


Random Forest...
   R² Score: 0.9447
   MAE: 0.0280


In [24]:
# 3. XGBoost
print("\nXGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
results['XGBoost'] = {
    'model': xgb_model,
    'r2': r2_score(y_test, xgb_pred),
    'mse': mean_squared_error(y_test, xgb_pred),
    'mae': mean_absolute_error(y_test, xgb_pred)
}
print(f"   R² Score: {results['XGBoost']['r2']:.4f}")
print(f"   MAE: {results['XGBoost']['mae']:.4f}")



XGBoost...
   R² Score: 0.9502
   MAE: 0.0270


In [25]:
# 4. LSTM (Fixed - By Stock)
print("\nLSTM Neural Network...")

# Prepare LSTM data for each stock separately
X_train_lstm_list = []
X_test_lstm_list = []
y_train_lstm_list = []
y_test_lstm_list = []

timesteps = 10

for ticker, df in processed_data.items():
    X_stock = df[feature_columns].values
    y_stock = df[target_column].values
    
    # Scale
    scaler_X_stock = MinMaxScaler()
    scaler_y_stock = MinMaxScaler()
    X_stock_scaled = scaler_X_stock.fit_transform(X_stock)
    y_stock_scaled = scaler_y_stock.fit_transform(y_stock.reshape(-1, 1)).ravel()
    
    # Create sequences
    X_seq = []
    y_seq = []
    for i in range(len(X_stock_scaled) - timesteps):
        X_seq.append(X_stock_scaled[i:i+timesteps])
        y_seq.append(y_stock_scaled[i+timesteps])
    
    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    
    # Split temporally
    split = int(len(X_seq) * train_ratio)
    X_train_lstm_list.append(X_seq[:split])
    X_test_lstm_list.append(X_seq[split:])
    y_train_lstm_list.append(y_seq[:split])
    y_test_lstm_list.append(y_seq[split:])

# Combine
X_train_lstm = np.vstack(X_train_lstm_list)
X_test_lstm = np.vstack(X_test_lstm_list)
y_train_lstm = np.concatenate(y_train_lstm_list)
y_test_lstm = np.concatenate(y_test_lstm_list)

print(f"LSTM Training samples: {len(X_train_lstm)}")
print(f"LSTM Testing samples: {len(X_test_lstm)}")

# Build LSTM
lstm_model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(timesteps, len(feature_columns))),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_test_lstm, y_test_lstm),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
lstm_pred = lstm_model.predict(X_test_lstm, verbose=0).flatten()
results['LSTM'] = {
    'model': lstm_model,
    'r2': r2_score(y_test_lstm, lstm_pred),
    'mse': mean_squared_error(y_test_lstm, lstm_pred),
    'mae': mean_absolute_error(y_test_lstm, lstm_pred)
}
print(f"   R² Score: {results['LSTM']['r2']:.4f}")
print(f"   MAE: {results['LSTM']['mae']:.4f}")


LSTM Neural Network...
LSTM Training samples: 8604
LSTM Testing samples: 2151
Epoch 1/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - loss: 0.0150 - mae: 0.0758 - val_loss: 0.0032 - val_mae: 0.0447
Epoch 2/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0023 - mae: 0.0351 - val_loss: 0.0025 - val_mae: 0.0374
Epoch 3/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0054 - val_mae: 0.0621
Epoch 4/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0015 - mae: 0.0289 - val_loss: 0.0048 - val_mae: 0.0584
Epoch 5/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.0014 - mae: 0.0281 - val_loss: 0.0023 - val_mae: 0.0374
Epoch 6/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.0013 - mae: 0.0267 - val_loss: 0.0019 - val_mae: 0.0332
Epoch 7/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0012 - mae: 0.0259 - val_loss: 0.0021 - val_mae: 0.0361
Epoch 8/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0011 - mae: 0.0250 - val

### 6. SELECT BEST MODEL

In [26]:
print("\n6. SELECTING BEST MODEL...")
print("-"*40)

# Find model with lowest MAE (more interpretable than R² for prices)
best_model_name = min(results, key=lambda x: results[x]['mae'])
best_model = results[best_model_name]['model']
best_mae = results[best_model_name]['mae']
best_r2 = results[best_model_name]['r2']

print(f"\nBEST MODEL: {best_model_name}")
print(f"   MAE: {best_mae:.4f}")
print(f"   R² Score: {best_r2:.4f}")

# Print all results
print("\nAll Models Performance:")
for name, metrics in results.items():
    print(f"   {name:20} R²: {metrics['r2']:.4f} | MAE: {metrics['mae']:.4f}")



6. SELECTING BEST MODEL...
----------------------------------------

BEST MODEL: Linear Regression
   MAE: 0.0178
   R² Score: 0.9760

All Models Performance:
   Linear Regression    R²: 0.9760 | MAE: 0.0178
   Random Forest        R²: 0.9447 | MAE: 0.0280
   XGBoost              R²: 0.9502 | MAE: 0.0270
   LSTM                 R²: 0.9398 | MAE: 0.0302


### 7. SAVE MODEL AND ASSETS

In [27]:
print("\n7. SAVING MODEL...")
print("-"*40)

# Save the best model
if best_model_name == 'LSTM':
    lstm_model.save('models/stock_predictor.h5')
    model_type = 'lstm'
else:
    with open('models/stock_predictor.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    model_type = 'sklearn'

# Save scalers
with open('models/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('models/scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

# Save feature list
with open('models/features.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

# Save metadata
metadata = {
    'best_model': best_model_name,
    'mae_score': best_mae,
    'r2_score': best_r2,
    'features': feature_columns,
    'model_type': model_type,
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'num_samples': len(processed_data),
    'all_results': {name: {'r2': m['r2'], 'mae': m['mae']} for name, m in results.items()}
}

with open('models/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)



7. SAVING MODEL...
----------------------------------------
